# 1) Dependencies

In [ ]:
%%capture # suppress output, unless there's an error
!pip install pandas numpy matplotlib tqdm
!pip install bokeh
!pip install plotly
!pip install nbformat --upgrade

# 2) Pointing to the Files (change as necessary)
and imports

Note the way I have this currently set up:
If it's in the directory with the data files, you just need to change file_template to match the .tec data files you're matching, and the number of files you expect.

In [ ]:
# Directory containing the files
data_dir = "."
file_template = "Erin_UC_increasedT-{:03d}.tec"
n_files = 100

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 3) Importing relevant details from the tec files

In [ ]:
# Storage for parsed data
all_data = []


# Function to extract data from one file
def read_tec_file(filepath):
    with open(filepath, "r") as f:
        lines = f.readlines()

    # Extract variables
    var_line = next(l for l in lines if l.startswith("VARIABLES"))
    variables = [v.strip().strip('"') for v in var_line.split("=")[1].split(",")]

    # Extract data lines (after the ZONE line)
    data_start_idx = next(i for i, l in enumerate(lines) if l.startswith("ZONE"))
    data_lines = lines[data_start_idx + 1 :]
    data = [list(map(float, l.strip().split())) for l in data_lines]

    df = pd.DataFrame(data, columns=variables)
    return df


# Read all files
for i in tqdm(range(n_files)):
    filename = file_template.format(i)
    filepath = os.path.join(data_dir, filename)
    df = read_tec_file(filepath)
    df["Time Index"] = i
    all_data.append(df)

In [ ]:
# combine them all together
full_df = pd.concat(all_data, ignore_index=True)

In [ ]:
def get_variable_names(filepath):
    with open(filepath, "r") as f:
        for line in f:
            if line.startswith("VARIABLES"):
                # Clean and split variable names
                variables = [
                    v.strip().strip('"') for v in line.split("=")[1].split(",")
                ]
                return variables


# Example usage
sample_file = "Erin_UC_increasedT-000.tec"
variable_names = get_variable_names(sample_file)

print("📊 Variables in Tecplot file:")
for i, var in enumerate(variable_names):
    print(f"{i:2d}: {var}")

# 4) Time dependent plot

In [ ]:
import plotly.express as px

# column names are variables so easier to change
ch4_column = "Total CH4(aq) [M]"
df_ch4 = full_df[["X [m]", "Y [m]", "Z [m]", ch4_column, "Time Index"]]

# want the legend to be consistent, so need to get max and min
# to think: logarithmic scale is necessary
ch4_min = df_ch4[ch4_column].min()
ch4_max = df_ch4[ch4_column].max()

# making the plot
fig = px.scatter_3d(
    df_ch4,
    x="X [m]",
    y="Y [m]",
    z="Z [m]",
    color=ch4_column,
    animation_frame="Time Index",  # This creates the slider
    color_continuous_scale="Viridis",
    range_color=[ch4_min, ch4_max],  # Fixed color scale
    labels={ch4_column: "CH4 Concentration [M]"},
    title="3D Visualization of CH4 Concentration Over Time",
    opacity=0.7,
)

# improved layout
fig.update_layout(
    scene=dict(xaxis_title="X [m]", yaxis_title="Y [m]", zaxis_title="Z [m]"),
    # time slider
    updatemenus=[
        {
            "type": "buttons",
            "showactive": False,
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [None, {"frame": {"duration": 100}}],
                },
                {
                    "label": "Pause",
                    "method": "animate",
                    "args": [[None], {"frame": {"duration": 0}, "mode": "immediate"}],
                },
            ],
        }
    ],
)

fig.show()

# 5) Multiple time dependent concentration plots

In [ ]:
# Define the variables you want to plot
variables_to_plot = [
    "Total CH4(aq) [M]",
    "Total O2(aq) [M]",
    "Total HCO3- [M]",
    "Total Fe+++ [M]",
]

# Filter available variables (in case some don't exist in your data)
available_vars = [var for var in variables_to_plot if var in full_df.columns]

# Create subplot grid (2x2 for 4 plots, adjust as needed)
n_plots = len(available_vars)
n_cols = 2
n_rows = (n_plots + n_cols - 1) // n_cols  # Ceiling division

# Create subplots with 3D scenes and more spacing
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    specs=[[{"type": "scatter3d"} for _ in range(n_cols)] for _ in range(n_rows)],
    subplot_titles=available_vars,
    vertical_spacing=0.15,
    horizontal_spacing=0.05,
)

# Get unique time indices for frames
time_indices = sorted(full_df["Time Index"].unique())

# Create frames for animation
frames = []
for time_idx in time_indices:
    frame_data = []
    time_data = full_df[full_df["Time Index"] == time_idx]

    for i, var in enumerate(available_vars):
        row = (i // n_cols) + 1
        col = (i % n_cols) + 1

        # Calculate consistent color scale for this variable
        var_min = full_df[var].min()
        var_max = full_df[var].max()

        # Better colorbar positioning to avoid overlap
        colorbar_x = (
            1.02 if col == n_cols else 0.48
        )  # Right edge for rightmost column, middle for left
        colorbar_y = (
            0.75 if row == 1 else 0.25
        )  # Top for first row, bottom for second row

        scatter = go.Scatter3d(
            x=time_data["X [m]"],
            y=time_data["Y [m]"],
            z=time_data["Z [m]"],
            mode="markers",
            marker=dict(
                size=3,
                color=time_data[var],
                colorscale="Viridis",
                cmin=var_min,
                cmax=var_max,
                colorbar=dict(
                    title=dict(text=f"{var.split(' ')[1]} [M]", font=dict(size=10)),
                    x=colorbar_x,
                    y=colorbar_y,
                    len=0.3,
                    thickness=15,
                    tickfont=dict(size=8),
                ),
                opacity=0.7,
            ),
            name=f"{var} (t={time_idx})",
        )
        frame_data.append(scatter)

    frames.append(go.Frame(data=frame_data, name=str(time_idx)))

# Add initial data (first time step)
initial_data = full_df[full_df["Time Index"] == time_indices[0]]
for i, var in enumerate(available_vars):
    row = (i // n_cols) + 1
    col = (i % n_cols) + 1

    var_min = full_df[var].min()
    var_max = full_df[var].max()

    # Better colorbar positioning
    colorbar_x = 1.02 if col == n_cols else 0.48
    colorbar_y = 0.75 if row == 1 else 0.25

    fig.add_trace(
        go.Scatter3d(
            x=initial_data["X [m]"],
            y=initial_data["Y [m]"],
            z=initial_data["Z [m]"],
            mode="markers",
            marker=dict(
                size=3,
                color=initial_data[var],
                colorscale="Viridis",
                cmin=var_min,
                cmax=var_max,
                colorbar=dict(
                    title=dict(text=f"{var.split(' ')[1]} [M]", font=dict(size=10)),
                    x=colorbar_x,
                    y=colorbar_y,
                    len=0.3,
                    thickness=15,
                    tickfont=dict(size=8),
                ),
                opacity=0.7,
            ),
            name=var,
        ),
        row=row,
        col=col,
    )

# Update layout with animation controls and better spacing
fig.update_layout(
    title="Multi-Variable 3D Concentration Visualization Over Time",
    height=900,  # Increased height
    width=1200,  # Set width for better proportions
    showlegend=False,
    margin=dict(l=50, r=150, t=80, b=100),  # More margin for colorbars
    updatemenus=[
        {
            "type": "buttons",
            "showactive": False,
            "x": 0.1,
            "y": 0.02,
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [None, {"frame": {"duration": 150}}],
                },
                {
                    "label": "Pause",
                    "method": "animate",
                    "args": [[None], {"frame": {"duration": 0}, "mode": "immediate"}],
                },
            ],
        }
    ],
    sliders=[
        {
            "steps": [
                {
                    "args": [[str(t)], {"frame": {"duration": 0}, "mode": "immediate"}],
                    "label": f"t={t}",
                    "method": "animate",
                }
                for t in time_indices
            ],
            "currentvalue": {"prefix": "Time Index: "},
            "len": 0.8,
            "x": 0.1,
            "y": 0.02,
        }
    ],
)

# Update 3D scene properties for all subplots
for i in range(1, n_plots + 1):
    fig.update_scenes(
        xaxis_title="X [m]",
        yaxis_title="Y [m]",
        zaxis_title="Z [m]",
        selector=dict(type="scene"),
    )

# Add frames to figure
fig.frames = frames

# Export to HTML
fig.write_html("multi_variable_concentration_3d.html")
print("Multi-variable 3D plot saved as 'multi_variable_concentration_3d.html'")

fig.show()